## The First Big Project: Professionally You!

### Introducing Tool Use with Pushover

**Pushover** is a simple yet powerful service for sending push notifications directly to your phone—perfect for monitoring your AI agents and workflows.

### Quick Setup Guide

1. **Create Your Account**  
   Visit [pushover.net](https://pushover.net/) and click **"Login or Signup"** in the top right corner.

2. **Generate API Credentials**  
   Once logged in, click **"Create an Application/API Token"** on your dashboard.  
   - Give it a name (e.g., "AI Agents" or "Lab Project")  
   - Click **Create Application**

3. **Add Keys to Your Environment**  
   Open your `.env` file and add these two lines:

PUSHOVER_USER=your_user_key_here_starts_with_u
PUSHOVER_TOKEN=your_app_token_here_starts_with_a

- **User Key**: Found on the top-right of your Pushover home screen (starts with `u`)  
- **App Token**: Found when you click into your newly created application (starts with `a`)

4. **Install the Mobile App**  
Click **"Add Phone, Tablet or Desktop"** and follow the instructions to install Pushover on your device.

Now you're ready to receive real-time notifications from your AI workflows!


In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
# === LLM SETUP (edit only this cell to change provider/model) ===
# Reads from project-root .env; defaults to Ollama.

load_dotenv(override=True)
openai = OpenAI(
    base_url=os.getenv("LOCAL_LLM_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.getenv("LOCAL_LLM_API_KEY", "ollama"),
)
model_name = os.getenv("LOCAL_LLM_MODEL", "llama3.2:latest")
print(f"Using model: {model_name}")

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [ ]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [ ]:
tools

In [ ]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
globals()["record_unknown_question"]("this is a really hard question")

In [ ]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Revanth Reddy T"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
def chat(message, history):
    push(f"New message from user: {message}")
    if not message.strip():
        return "Please type something to start our conversation!"

    system = system_prompt
    # if contains_sensitive_content(message):
    #     system = system_prompt + "\n\n" + COMPLIANCE_ADDENDUM

    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    done = False
    for _ in range(2): # Allow for one retry after truncation
        try:
            response = openai.chat.completions.create(model=model_name, messages=messages, tools=tools)
            finish_reason = response.choices[0].finish_reason

            if finish_reason=="tool_calls":
                message = response.choices[0].message
                tool_calls = message.tool_calls
                results = handle_tool_calls(tool_calls)
                messages.append(message)
                messages.extend(results)
            else:
                done = True
                return response.choices[0].message.content

        except openai.APIError as e:
            print(f"LLM API Error: {e}")
            if len(history) > 5: # Truncate to last 5 messages
                history = history[-5:]
                messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
                print("History truncated due to API error. Retrying with shorter history.")
            else:
                return "I encountered an issue processing your request, and even after adjusting, I'm still having trouble. Perhaps we can start a new conversation?"
    return "I'm sorry, I'm currently unable to process your request. Please try again later."

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

# Project 1 Deployment Requirements (HuggingFace Spaces)

## Preparation before deployment:
- Update files in your **"me" directory** (e.g., LinkedIn profile, summary.txt) to personalize with your information.
- Ensure there is **no README file inside the `1_foundations` directory**; delete it if present.

## Steps to deploy:
1. **Create a HuggingFace account** at https://huggingface.co if you don’t have one.
2. Generate an **Access Token** with WRITE permissions:
   - From the user avatar menu (top right), select **Access Tokens** → **Create New Token** → set permissions to WRITE.
   - Add this token in your local `.env` file as `HF_TOKEN=hf_xxx`.
3. From the `1_foundations` folder, deploy the app using:
   - `uv run gradio deploy`
   - If it still prompts for the token, interrupt (Ctrl+C) and run:
     ```
     uv run dotenv -f ../.env run -- uv run gradio deploy
     ```
     This forces environment variables to load correctly.
4. During deployment, provide:
   - Space name: **career_conversation**
   - Application file: `app.py`
   - Hardware: **cpu-basic**
   - Confirm needing to supply secrets
   - Supply secret keys for:
     - OpenAI API key (`OPENAI_API_KEY`)
     - Pushover user key and token (`PUSHOVER_USER` and `PUSHOVER_TOKEN`)
   - Decline GitHub Actions integration (say **No**).

## Troubleshooting HuggingFace Token issues:
- Restart your environment or terminal.
- Run `load_dotenv(override=True)` in your code or start a new terminal session.
- In the terminal, set the token manually with:
  - Powershell: `$env:HF_TOKEN = "hf_XXXX"`
  - Bash: `export HF_TOKEN=hf_XXXX`

## Managing secrets after deployment:
- Log in to HuggingFace website.
- Open your deployed Space via your profile menu.
- Access Space **Settings** (gear icon, top right).
- Update or delete secrets as needed from the Settings interface.

## Other notes:
- The deployment process generates a new README file in `1_foundations`.
- To redeploy cleanly, delete this README so the deploy prompts will reappear.
- You can delete your Space anytime using the web UI in the Space settings.

---

## Optional improvements / exercises:
- Add more personal context/resources (e.g., a knowledge base using Retrieval-Augmented Generation).
- Enhance with additional tools like SQL databases for Q&A integration.
- Incorporate agentic patterns or evaluation tools from prior labs.
- Deploy for personal use as a career assistant tool or build business applications with AI assistants using domain-specific expertise.
